In [3]:
import os
import sys
import torch as t
import pandas as pd

PROJECT_ROOT = r"C:\Mestrado\Graph_Pruning\AnyGraph"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
sys.argv = ["analysis"]

In [4]:
from model import AnyGraph
from params import args

In [5]:
def get_linear_weights(model):
    weights = {}

    for name, param in model.named_parameters():
        if (
            "trainable_nn.dense_layers" in name and
            "linear.weight" in name
        ):
            weights[name] = param.detach().cpu().flatten()

    return weights


In [6]:
def weight_statistics(weight_dict):

    rows = []

    for name, w in weight_dict.items():

        abs_w = w.abs()

        rows.append({

            "Layer": name,

            "Num Weights": w.numel(),

            "Mean": w.mean().item(),

            "Std": w.std().item(),

            "Median": w.median().item(),

            "Min": w.min().item(),

            "Max": w.max().item(),

            "|w| < 1e-3 (%)":
                (abs_w < 1e-3).float().mean().item()*100,

            "|w| < 1e-4 (%)":
                (abs_w < 1e-4).float().mean().item()*100,

            "|w| < 1e-5 (%)":
                (abs_w < 1e-5).float().mean().item()*100

        })

    return pd.DataFrame(rows)

In [7]:
def load_model_from_checkpoint(checkpoint_path):
    #Configuração mínima que o model.py espera
    args.devices = ["cuda:0", "cuda:0"]

    #Carregando o modelo
    checkpoint = t.load(f"Models/{checkpoint_path}.mod", weights_only=False)
    model = checkpoint["model"]
    return model 

Analise do modelo pretreinado 1

In [17]:
model1 = load_model_from_checkpoint("pretrain_link1")
print(type(model1))

<class 'model.AnyGraph'>


In [ ]:
model1.named_parameters()

model1.named_modules()

model1.experts

#model.trainable_nn

ModuleList(
  (0-7): 8 x Expert(
    (topo_encoder): TopoEncoder(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=False)
    )
    (trainable_nn): MLP(
      (dense_layers): Sequential(
        (0): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (1): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (2): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (3): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (4): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (5): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)


In [20]:
for name, param in model1.named_parameters():
    print(name, param.shape)

experts.0.trainable_nn.dense_layers.0.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.0.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.1.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.1.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.2.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.2.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.3.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.3.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.4.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.4.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.5.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.5.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.6.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.6.linear.bias torch.Size([512])

In [ ]:
for name, param in model1.named_parameters():
    if "linear.weight" in name:
        print(name, param.shape)

experts.0.trainable_nn.dense_layers.0.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.1.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.2.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.3.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.4.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.5.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.6.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.7.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.0.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.1.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.2.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.3.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.4.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.de

In [ ]:
for i, expert in enumerate(model1.experts):
    total = sum(p.numel() for p in expert.parameters())
    print(f"Expert {i}: {total:,} parâmetros")

Expert 0: 2,109,440 parâmetros
Expert 1: 2,109,440 parâmetros
Expert 2: 2,109,440 parâmetros
Expert 3: 2,109,440 parâmetros
Expert 4: 2,109,440 parâmetros
Expert 5: 2,109,440 parâmetros
Expert 6: 2,109,440 parâmetros
Expert 7: 2,109,440 parâmetros


In [ ]:
for i, expert in enumerate(model1.experts):
    linear_total = 0
    for name, p in expert.named_parameters():
        if "linear.weight" in name:
            linear_total += p.numel()

    print(f"Expert {i}: {linear_total:,} pesos lineares")

Expert 0: 2,097,152 pesos lineares
Expert 1: 2,097,152 pesos lineares
Expert 2: 2,097,152 pesos lineares
Expert 3: 2,097,152 pesos lineares
Expert 4: 2,097,152 pesos lineares
Expert 5: 2,097,152 pesos lineares
Expert 6: 2,097,152 pesos lineares
Expert 7: 2,097,152 pesos lineares


Esse Anygraph é formado por 8 experts simetricos, compostos por mlps

Total = 2.109.440

Lineares = 2.097.152

vieses e layerNorms = 12.288

2.097.152 / 2.109.440 ≈ 99,42%

99,4% dos parâmetros treináveis de cada Expert estão nas matrizes das camadas Lineares.

In [30]:
weights1 = get_linear_weights(model1)
weights1.keys()

dict_keys(['experts.0.trainable_nn.dense_layers.0.linear.weight', 'experts.0.trainable_nn.dense_layers.1.linear.weight', 'experts.0.trainable_nn.dense_layers.2.linear.weight', 'experts.0.trainable_nn.dense_layers.3.linear.weight', 'experts.0.trainable_nn.dense_layers.4.linear.weight', 'experts.0.trainable_nn.dense_layers.5.linear.weight', 'experts.0.trainable_nn.dense_layers.6.linear.weight', 'experts.0.trainable_nn.dense_layers.7.linear.weight', 'experts.1.trainable_nn.dense_layers.0.linear.weight', 'experts.1.trainable_nn.dense_layers.1.linear.weight', 'experts.1.trainable_nn.dense_layers.2.linear.weight', 'experts.1.trainable_nn.dense_layers.3.linear.weight', 'experts.1.trainable_nn.dense_layers.4.linear.weight', 'experts.1.trainable_nn.dense_layers.5.linear.weight', 'experts.1.trainable_nn.dense_layers.6.linear.weight', 'experts.1.trainable_nn.dense_layers.7.linear.weight', 'experts.2.trainable_nn.dense_layers.0.linear.weight', 'experts.2.trainable_nn.dense_layers.1.linear.weight',

In [31]:
df_stats1 = weight_statistics(weights1)

df_stats1.head()

,Layer,Num Weights,Mean,Std,Median,Min,Max,|w| < 1e-3 (%),|w| < 1e-4 (%),|w| < 1e-5 (%)
0,experts.0.trainable_nn.dense_layers.0.linear.w...,262144,0.001117,0.026843,0.000947,-0.343339,0.531635,4.100800,0.406647,0.043106
1,experts.0.trainable_nn.dense_layers.1.linear.w...,262144,0.001863,0.030499,0.001673,-0.328579,0.356440,2.759171,0.271606,0.028610
2,experts.0.trainable_nn.dense_layers.2.linear.w...,262144,0.001534,0.030954,0.001534,-0.617633,0.334790,2.533340,0.238419,0.019836
3,experts.0.trainable_nn.dense_layers.3.linear.w...,262144,0.001315,0.027611,0.000601,-0.352280,0.281027,6.935883,1.473236,0.410080
4,experts.0.trainable_nn.dense_layers.4.linear.w...,262144,0.001299,0.022279,0.000123,-0.414425,0.269411,18.498993,7.813644,4.129028


In [32]:
expert_rows = []

for expert in range(8):

    tensors = []

    for layer in range(8):

        tensors.append(
            weights1[
                f"experts.{expert}.trainable_nn.dense_layers.{layer}.linear.weight"
            ]
        )

    W = t.cat(tensors)

    absW = W.abs()

    expert_rows.append({

        "Expert": expert,

        "Weights": W.numel(),

        "Mean": W.mean().item(),

        "Std": W.std().item(),

        "Median": W.median().item(),

        "Min": W.min().item(),

        "Max": W.max().item(),
        "L1 Norm": absW.sum().item(),

        "L2 Norm": t.norm(W).item(),

        "<1e-3 (%)":
            (absW < 1e-3).float().mean().item()*100,

        "<1e-4 (%)":
            (absW < 1e-4).float().mean().item()*100,

        "<1e-5 (%)":
            (absW < 1e-5).float().mean().item()*100,

    })

expert_df = pd.DataFrame(expert_rows)

expert_df

,Expert,Weights,Mean,Std,Median,Min,Max,L1 Norm,L2 Norm,<1e-3 (%),<1e-4 (%),<1e-5 (%)
0,0,2097152,0.001262,0.023709,4.007663e-05,-0.617633,0.531635,32453.343750,34.377640,18.096685,9.848833,7.106161
1,1,2097152,0.000447,0.024989,3.512960e-04,-0.484323,0.293547,40152.984375,36.189846,4.682255,0.471878,0.046921
2,2,2097152,0.000773,0.023183,2.652669e-07,-0.478156,0.451406,31587.052734,33.585968,20.472145,12.181044,9.259272
3,3,2097152,0.000733,0.024002,3.129148e-04,-0.464606,0.251190,36194.058594,34.768135,9.354877,0.985527,0.100327
4,4,2097152,0.000579,0.025751,5.153752e-04,-0.502156,0.216051,43107.406250,37.299397,3.049946,0.302887,0.030422
5,5,2097152,0.001170,0.024845,6.082886e-04,-0.386063,0.332546,38912.500000,36.014225,6.619215,0.678205,0.069380
6,6,2097152,0.000590,0.025337,4.739435e-04,-0.382096,0.289678,41462.546875,36.698757,3.625631,0.360441,0.036049
7,7,2097152,0.001146,0.028890,4.311657e-07,-0.548574,0.679273,38183.113281,41.865173,22.649765,14.181662,11.276627


In [ ]:
all_weights1 = t.cat(list(weights1.values()))

absW = all_weights1.abs()

global_stats = {

    "Num Weights":
        all_weights1.numel(),

    "Mean":
        all_weights1.mean().item(),

    "Std":
        all_weights1.std().item(),

    "Median":
        all_weights1.median().item(),

    "Min":
        all_weights1.min().item(),

    "Max":
        all_weights1.max().item(),

    "<1e-3 (%)":
        (absW < 1e-3).float().mean().item()*100,

    "<1e-4 (%)":
        (absW < 1e-4).float().mean().item()*100,

    "<1e-5 (%)":
        (absW < 1e-5).float().mean().item()*100

}

print(global_stats)

Analise do modelo pretreinado 2

In [8]:
model2 = load_model_from_checkpoint("pretrain_link2")
print(type(model2))

<class 'model.AnyGraph'>


In [9]:
model2.named_parameters()

model2.named_modules()

model2.experts

ModuleList(
  (0-7): 8 x Expert(
    (topo_encoder): TopoEncoder(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=False)
    )
    (trainable_nn): MLP(
      (dense_layers): Sequential(
        (0): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (1): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (2): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (3): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (4): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)
          (act): ReLU()
        )
        (5): FeedForwardLayer(
          (linear): Linear(in_features=512, out_features=512, bias=True)


In [10]:
for name, param in model2.named_parameters():
    print(name, param.shape)

experts.0.trainable_nn.dense_layers.0.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.0.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.1.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.1.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.2.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.2.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.3.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.3.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.4.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.4.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.5.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.5.linear.bias torch.Size([512])
experts.0.trainable_nn.dense_layers.6.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.6.linear.bias torch.Size([512])

In [11]:
for name, param in model2.named_parameters():
    if "linear.weight" in name:
        print(name, param.shape)

experts.0.trainable_nn.dense_layers.0.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.1.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.2.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.3.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.4.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.5.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.6.linear.weight torch.Size([512, 512])
experts.0.trainable_nn.dense_layers.7.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.0.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.1.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.2.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.3.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.dense_layers.4.linear.weight torch.Size([512, 512])
experts.1.trainable_nn.de

In [12]:
for i, expert in enumerate(model2.experts):
    total = sum(p.numel() for p in expert.parameters())
    print(f"Expert {i}: {total:,} parâmetros")

Expert 0: 2,109,440 parâmetros
Expert 1: 2,109,440 parâmetros
Expert 2: 2,109,440 parâmetros
Expert 3: 2,109,440 parâmetros
Expert 4: 2,109,440 parâmetros
Expert 5: 2,109,440 parâmetros
Expert 6: 2,109,440 parâmetros
Expert 7: 2,109,440 parâmetros


In [13]:
for i, expert in enumerate(model2.experts):
    linear_total = 0
    for name, p in expert.named_parameters():
        if "linear.weight" in name:
            linear_total += p.numel()

    print(f"Expert {i}: {linear_total:,} pesos lineares")

Expert 0: 2,097,152 pesos lineares
Expert 1: 2,097,152 pesos lineares
Expert 2: 2,097,152 pesos lineares
Expert 3: 2,097,152 pesos lineares
Expert 4: 2,097,152 pesos lineares
Expert 5: 2,097,152 pesos lineares
Expert 6: 2,097,152 pesos lineares
Expert 7: 2,097,152 pesos lineares


In [14]:
weights2 = get_linear_weights(model2)
weights2.keys()

dict_keys(['experts.0.trainable_nn.dense_layers.0.linear.weight', 'experts.0.trainable_nn.dense_layers.1.linear.weight', 'experts.0.trainable_nn.dense_layers.2.linear.weight', 'experts.0.trainable_nn.dense_layers.3.linear.weight', 'experts.0.trainable_nn.dense_layers.4.linear.weight', 'experts.0.trainable_nn.dense_layers.5.linear.weight', 'experts.0.trainable_nn.dense_layers.6.linear.weight', 'experts.0.trainable_nn.dense_layers.7.linear.weight', 'experts.1.trainable_nn.dense_layers.0.linear.weight', 'experts.1.trainable_nn.dense_layers.1.linear.weight', 'experts.1.trainable_nn.dense_layers.2.linear.weight', 'experts.1.trainable_nn.dense_layers.3.linear.weight', 'experts.1.trainable_nn.dense_layers.4.linear.weight', 'experts.1.trainable_nn.dense_layers.5.linear.weight', 'experts.1.trainable_nn.dense_layers.6.linear.weight', 'experts.1.trainable_nn.dense_layers.7.linear.weight', 'experts.2.trainable_nn.dense_layers.0.linear.weight', 'experts.2.trainable_nn.dense_layers.1.linear.weight',

In [15]:
df_stats2 = weight_statistics(weights2)

df_stats2.head()

,Layer,Num Weights,Mean,Std,Median,Min,Max,|w| < 1e-3 (%),|w| < 1e-4 (%),|w| < 1e-5 (%)
0,experts.0.trainable_nn.dense_layers.0.linear.w...,262144,0.004288,0.032987,0.003638,-0.301945,0.283323,2.735901,0.267029,0.024796
1,experts.0.trainable_nn.dense_layers.1.linear.w...,262144,0.004123,0.033851,0.003884,-0.323326,0.313263,2.478027,0.254059,0.027466
2,experts.0.trainable_nn.dense_layers.2.linear.w...,262144,0.003045,0.034087,0.002918,-0.376504,0.491927,2.508926,0.263977,0.030899
3,experts.0.trainable_nn.dense_layers.3.linear.w...,262144,0.002718,0.030501,0.001910,-0.507507,0.636501,4.553223,0.813675,0.285721
4,experts.0.trainable_nn.dense_layers.4.linear.w...,262144,0.002166,0.026524,0.001238,-0.672991,0.472274,8.659363,3.016663,1.537323


In [16]:
expert_rows = []

for expert in range(8):

    tensors = []

    for layer in range(8):

        tensors.append(
            weights2[
                f"experts.{expert}.trainable_nn.dense_layers.{layer}.linear.weight"
            ]
        )

    W = t.cat(tensors)

    absW = W.abs()

    expert_rows.append({

        "Expert": expert,

        "Weights": W.numel(),

        "Mean": W.mean().item(),

        "Std": W.std().item(),

        "Median": W.median().item(),

        "Min": W.min().item(),

        "Max": W.max().item(),
        "L1 Norm": absW.sum().item(),

        "L2 Norm": t.norm(W).item(),

        "<1e-3 (%)":
            (absW < 1e-3).float().mean().item()*100,

        "<1e-4 (%)":
            (absW < 1e-4).float().mean().item()*100,

        "<1e-5 (%)":
            (absW < 1e-5).float().mean().item()*100,

    })

expert_df = pd.DataFrame(expert_rows)

expert_df

,Expert,Weights,Mean,Std,Median,Min,Max,L1 Norm,L2 Norm,<1e-3 (%),<1e-4 (%),<1e-5 (%)
0,0,2097152,0.002473,0.027715,2.125106e-04,-0.672991,0.636501,39052.976562,40.292488,13.693666,8.583117,7.001448
1,1,2097152,0.000489,0.023456,2.747468e-26,-0.489035,0.718215,28462.132812,33.970692,34.272861,27.810717,25.536013
2,2,2097152,0.001205,0.023416,4.414230e-27,-0.483083,0.561250,26471.851562,33.952019,43.052006,37.180614,34.713984
3,3,2097152,0.000430,0.023534,9.639402e-28,-0.500799,0.391589,28595.517578,34.083546,38.721561,33.227873,31.244898
4,4,2097152,0.000270,0.022902,7.808905e-25,-0.480366,0.428917,28072.400391,33.165264,36.816359,30.448532,27.990341
5,5,2097152,0.000564,0.027594,3.709266e-24,-0.531456,0.608848,33212.433594,39.966148,34.719372,29.096556,26.866150
6,6,2097152,0.001167,0.023853,1.066684e-16,-0.475818,0.479662,28375.302734,34.579723,33.745289,26.498032,23.372602
7,7,2097152,0.001227,0.026331,1.305078e-21,-0.517033,0.550757,32267.873047,38.168785,33.144951,27.043724,24.549294


In [18]:
all_weights2 = t.cat(list(weights2.values()))

absW = all_weights2.abs()

global_stats = {

    "Num Weights":
        all_weights2.numel(),

    "Mean":
        all_weights2.mean().item(),

    "Std":
        all_weights2.std().item(),

    "Median":
        all_weights2.median().item(),

    "Min":
        all_weights2.min().item(),

    "Max":
        all_weights2.max().item(),

    "<1e-3 (%)":
        (absW < 1e-3).float().mean().item()*100,

    "<1e-4 (%)":
        (absW < 1e-4).float().mean().item()*100,

    "<1e-5 (%)":
        (absW < 1e-5).float().mean().item()*100

}

global_stats

{'Num Weights': 16777216,
 'Mean': 0.0009780421387404203,
 'Std': 0.02493058145046234,
 'Median': 2.888304395028728e-23,
 'Min': -0.6729912757873535,
 'Max': 0.7182154059410095,
 '<1e-3 (%)': 33.52075815200806,
 '<1e-4 (%)': 27.486145496368408,
 '<1e-5 (%)': 25.15934109687805}